In [1]:
import tuner_functions

In [2]:
tuner_functions.initialize_tensors()

Made embeddings in 255.63 seconds
Made continuous tensor in 40.75 seconds
Made results tensor in 10.82 seconds
Made phase tensor in 5.80 seconds


In [5]:
run_data = {}
tuner4 = tuner_functions.get_model(
    optimizer = "lbfgs",
    l2_lambda=1e-3,
    run_data = run_data
)

epoch 1; t=154; bce_loss=0.6931; loss=0.6931; max(∆w) = 104.1366
epoch 2; t=298; bce_loss=0.5257; loss=0.5257; max(∆w) = 313.0186
epoch 3; t=442; bce_loss=0.4910; loss=0.4916; max(∆w) = 344.9004
epoch 4; t=585; bce_loss=0.4633; loss=0.4651; max(∆w) = 534.3229
epoch 5; t=724; bce_loss=0.4347; loss=0.4385; max(∆w) = 462.7726
epoch 6; t=863; bce_loss=0.4182; loss=0.4225; max(∆w) = 224.9029
epoch 7; t=1000; bce_loss=0.4139; loss=0.4178; max(∆w) = 137.3012
epoch 8; t=1141; bce_loss=0.4119; loss=0.4154; max(∆w) = 156.6686
epoch 9; t=1277; bce_loss=0.4100; loss=0.4129; max(∆w) = 277.8923
epoch 10; t=1417; bce_loss=0.4076; loss=0.4096; max(∆w) = 302.3549
epoch 11; t=1552; bce_loss=0.4050; loss=0.4065; max(∆w) = 198.6160
epoch 12; t=1687; bce_loss=0.4036; loss=0.4048; max(∆w) = 131.6836
epoch 13; t=1825; bce_loss=0.4031; loss=0.4041; max(∆w) = 83.3997
epoch 14; t=1959; bce_loss=0.4024; loss=0.4033; max(∆w) = 93.9595
epoch 15; t=2095; bce_loss=0.4019; loss=0.4026; max(∆w) = 105.6557
epoch 16; t=

In [6]:
tuner_functions.print_continuous_weights(tuner4)

constexpr packed_eval_t mobility[6] = {S(-13, -25), S(1, 5), S(9, 12), S(8, 6), S(3, 10), S(-19, 1)};
// hash=2c7dcade45139c90877f774d2aff1c1c5eaccadf6c97b08f426dda8fda187db1


In [7]:
tuner_functions.print_king_bucketed_psts(tuner4)

    constexpr std::array<std::array<std::array<packed_eval_t, 64>, 6>, 2> psts = {{
          {{
                   {
                       S(171,349), S(171,349), S(171,349), S(171,349), S(171,349), S(171,349), S(171,349), S(171,349), 
                       S(166,247), S(248,262), S(199,249), S(134,280), S( 98,286), S(105,277), S( 74,303), S( 15,284), 
                       S(175,235), S(203,260), S(189,239), S(123,272), S(136,253), S(104,254), S( 73,271), S( 14,256), 
                       S(153,270), S(196,287), S(184,254), S(168,263), S(154,253), S(125,252), S(109,274), S( 28,254), 
                       S(171,332), S(207,344), S(189,305), S(149,295), S(177,271), S(159,262), S(125,299), S( 50,289), 
                       S(188,486), S(206,536), S(182,491), S(199,431), S(204,408), S(206,358), S(157,418), S( 61,398), 
                       S(228,593), S(298,519), S(254,614), S(256,528), S(212,507), S(199,466), S(164,501), S( 86,517), 
                       S(171,349), S(171,3

In [8]:
tuner_functions.save_eval_to_file(tuner4, "NoahResults.txt")

len(evals) is 7153281


In [30]:
import torch
from tuner_functions import Tuner, NUM_KING_BUCKETS
eval_t_name = "packed_eval_t"
def get_material_weights(model : Tuner):
    material_weights = []
    params = model.embedding_layers
    for piece_type in range(6):
        raw_weights = params[piece_type].weight.data[:-1,:]
        material_weights.append(
            torch.round(raw_weights[:-1,:].mean(axis = 0)).flatten().int()
        )
    return material_weights

def print_material_weights(model : Tuner):
    w = get_material_weights(model)
    strw = [f"S{int(e[0]), int(e[1])}" for e in w]
    strw2 = "{" + ", ".join(strw) + "}"
    line = f"constexpr {eval_t_name} material[6] = {strw2};"
    print(line)

def print_king_bucketed_psts(model: Tuner):
    params = model.embedding_layers
    material_weights = get_material_weights(model)
    print(f"    constexpr std::array<std::array<std::array<{eval_t_name}, 64>, 6>, {NUM_KING_BUCKETS}> psts = " + "{{")
    for bucket in range(NUM_KING_BUCKETS):
        idx = bucket * 64
        print(" " * 10 + "{{")
        for piece_type in range(6):
            print(" " * 19 + "{")

            # Get PSTs for just this bucket
            raw_weights = params[piece_type].weight.data.int()[:-1,:]
            pst_weights = raw_weights - material_weights[piece_type]
            pst_weights_bucketed = pst_weights[idx:idx+64,:]

            # Determine how much whitespace we need
            l1 = len(str(torch.min(pst_weights_bucketed).item()))
            l2 = len(str(torch.max(pst_weights_bucketed).item()))
            l = max(l1, l2)

            for i, thing in enumerate(pst_weights_bucketed):
                print(f"{" " * 23 * (1 - i % 8)}S({thing[0]:{l}},{thing[1]:{l}}), ", end = "\n" * (i % 8 - 6))
            print(" " * 19 + "},")
        print(" " * 10 + "}},")
    print(" " * 8 + "}}; // hash=" + f"{model.get_hash()}")

In [31]:
print_material_weights(tuner4)

constexpr packed_eval_t material[6] = {S(163, 339), S(470, 600), S(489, 584), S(584, 1107), S(996, 2141), S(-1, 1)};


In [32]:
print_king_bucketed_psts(tuner4)

    constexpr std::array<std::array<std::array<packed_eval_t, 64>, 6>, 2> psts = {{
          {{
                   {
                       S(   8,  10), S(   8,  10), S(   8,  10), S(   8,  10), S(   8,  10), S(   8,  10), S(   8,  10), S(   8,  10), 
                       S(   3, -92), S(  85, -77), S(  36, -90), S( -29, -59), S( -65, -53), S( -58, -62), S( -89, -36), S(-148, -55), 
                       S(  12,-104), S(  40, -79), S(  26,-100), S( -40, -67), S( -27, -86), S( -59, -85), S( -90, -68), S(-149, -83), 
                       S( -10, -69), S(  33, -52), S(  21, -85), S(   5, -76), S(  -9, -86), S( -38, -87), S( -54, -65), S(-135, -85), 
                       S(   8,  -7), S(  44,   5), S(  26, -34), S( -14, -44), S(  14, -68), S(  -4, -77), S( -38, -40), S(-113, -50), 
                       S(  25, 147), S(  43, 197), S(  19, 152), S(  36,  92), S(  41,  69), S(  43,  19), S(  -6,  79), S(-102,  59), 
                       S(  65, 254), S( 135, 180), S(  91, 275), S